# Results comparison

Reads the outputs of `scripts/evaluate.py` (`artifacts/<dataset>/<arch>/<orig>/results/`).
No training happens here. Pull results from the cluster first:

```bash
bash cluster/sync.sh pull-results
```

Metric definitions: see `document.md`, section "Metrics".

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

repo_root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
results = sorted((repo_root / "artifacts").glob("*/*/*/results/summary.csv"))
for r in results:
    print(r.relative_to(repo_root))

In [ ]:
# Pick one experiment (dataset / arch / original)
summary_path = results[0]
summary = pd.read_csv(summary_path)
per_run = pd.read_csv(summary_path.parent / "per_run.csv")
display(Markdown((summary_path.parent / "summary.md").read_text()))

## Forgetting quality vs. utility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.scatter(summary["forget_quality_F"], summary["test_acc"])
for _, r in summary.iterrows():
    ax.annotate(r["name"], (r["forget_quality_F"], r["test_acc"]), fontsize=8)
ax.axhline(summary["retrain_test_acc"].iloc[0], ls="--", c="gray", label="retrain TA")
ax.set_xlabel("forgetting quality F (higher = closer to retrain)"); ax.set_ylabel("test accuracy"); ax.legend()

ax = axes[1]
ax.scatter(summary["seconds"], summary["competition_score"])
for _, r in summary.iterrows():
    ax.annotate(r["name"], (r["seconds"], r["competition_score"]), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("unlearning time per run (s)"); ax.set_ylabel("competition score")
plt.tight_layout()

## Per-run spread (forget accuracy, MIA) against the retrained oracle

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
order = ["retrain"] + list(summary.sort_values("competition_score", ascending=False)["name"])
for ax, col in zip(axes, ["forget_acc", "mia_forget_vs_test", "mia_efficacy"]):
    data = [per_run.loc[per_run["name"] == n, col] for n in order]
    ax.boxplot(data, vert=False, labels=order)
    ax.set_title(col)
plt.tight_layout()

## Plots saved by evaluate.py

In [ ]:
for name in ["competition_score.png", "epsilon_histograms.png", "loss_histograms_run0.png"]:
    p = summary_path.parent / "plots" / name
    if p.exists():
        display(Markdown(f"**{name}**")); display(Image(str(p)))